# 04. Auto-play with the trained model & creating the submission files

Put the `.pt` file you downloaded in `03_train_bc.ipynb` into `../models/` before you run this notebook.

The AI's play (1 episode) and the submission files are made in the same run. The video, CSV files, and zip are all saved in `results/`.

In [ ]:
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from nes_py.wrappers import JoypadSpace

# Competition rules shared with 01/02. Do not edit that file.
from competition_config import TARGET_FPS

ENV_ID = "SuperMarioBros-1-1-v0"

env = JoypadSpace(gym_super_mario_bros.make(ENV_ID, render_mode="human"), SIMPLE_MOVEMENT)

# Same decimation as 01_collect_data.ipynb. The training data updates its frame
# history at TARGET_FPS, so inference picks an action at the same rate and
# holds it in between (frame skip / action repeat).
RECORD_EVERY_N = round(env.metadata["render_fps"] / TARGET_FPS)

print("SIMPLE_MOVEMENT:", SIMPLE_MOVEMENT)
print("make sure this ordering matches the one used in 03_train_bc.ipynb")
print(f"render_fps={env.metadata['render_fps']} -> inference fps={TARGET_FPS} (RECORD_EVERY_N={RECORD_EVERY_N})")

In [ ]:
# --- Keep this block identical in 03_train_bc.ipynb and 04_playagent.ipynb ---
from collections import deque

import cv2
import numpy as np
import torch
import torch.nn as nn

FRAME_SIZE = (84, 84)

# --- STACK_SIZE: how many consecutive frames the model sees at once -----
# 1 turns this into an ordinary single-image classifier; 4 is the usual choice
# in the Atari literature. Try values in the 1-4 range and compare how well the
# agent actually plays.
#
# A single frame cannot show velocity. Whether Mario is rising or falling in a
# jump, and how fast he is moving, only becomes visible across several frames.
# Stacking more frames makes that observable, at the cost of a larger input and
# more memory (the dataset holds STACK_SIZE copies of every frame).
#
# Set this in 03 only. 04_playagent.ipynb reads the value back out of the
# checkpoint, so there is nothing to keep in sync there by hand.
STACK_SIZE = 4
# ---------------------------------------------------------------------------

NUM_ACTIONS = len(SIMPLE_MOVEMENT)


def preprocess_frame(frame: np.ndarray) -> np.ndarray:
    """Convert an RGB frame to grayscale and resize it to 84x84."""
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    return cv2.resize(gray, FRAME_SIZE, interpolation=cv2.INTER_AREA)


class FrameStacker:
    """Stack the most recent STACK_SIZE preprocessed frames."""

    def __init__(self, stack_size: int = STACK_SIZE):
        self.stack_size = stack_size
        self.frames = deque(maxlen=stack_size)

    def reset(self, processed_frame: np.ndarray) -> np.ndarray:
        self.frames.clear()
        for _ in range(self.stack_size):
            self.frames.append(processed_frame)
        return self.get()

    def step(self, processed_frame: np.ndarray) -> np.ndarray:
        self.frames.append(processed_frame)
        return self.get()

    def get(self) -> np.ndarray:
        return np.stack(self.frames, axis=0)


class MarioCNN(nn.Module):

    # Constructor: this runs when we make a model.
    # Unlike Model() in Week05, it takes two arguments: the number of stacked
    # frames (input channels) and the number of actions (outputs).
    def __init__(self, in_channels=STACK_SIZE, num_actions=NUM_ACTIONS):
        super(MarioCNN, self).__init__()   # do not change

        # "features": find the patterns in the frames.
        # Each input channel is one grayscale frame, so in_channels=STACK_SIZE
        # (it was 3 for the RGB colors in Week05). The frames (84x84) are larger
        # than the CIFAR10 images (32x32), so there is one more Conv2d + MaxPool2d
        # block than in Week05: 84x84 becomes 42x42, then 21x21, then 10x10.
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        # "junction": make a fixed size, then change it into 1D data
        self.junction = nn.Sequential(
            nn.AdaptiveAvgPool2d(output_size=(8, 8)),
            nn.Flatten()
        )

        # "classifier": decide the action from the features
        self.classifier = nn.Sequential(
            nn.Linear(in_features=64*8*8, out_features=256),
            nn.ReLU(),
            nn.Linear(in_features=256, out_features=num_actions)
        )

    # forward(): this runs when we give data to the model
    def forward(self, x):
        x = x.float() / 255.0   # frames are 0-255 integers; in Week05, ToTensor() did this
        x = self.features(x)
        x = self.junction(x)
        x = self.classifier(x)
        return x
# --- end of shared block ---

In [ ]:
from pathlib import Path

# change this to the file you downloaded from 03_train_bc.ipynb
CHECKPOINT_PATH = Path("../models/mario_bc_s4_YYYYMMDD_HHMMSS.pt")

device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
    STACK_SIZE = checkpoint["stack_size"]
else:
    state_dict = checkpoint
    STACK_SIZE = state_dict["features.0.weight"].shape[1]
    print(f"no metadata in this checkpoint - inferred STACK_SIZE={STACK_SIZE} from its weights")

model = MarioCNN(in_channels=STACK_SIZE).to(device)
model.load_state_dict(state_dict)
model.eval()
print(f"loaded: {CHECKPOINT_PATH} (device={device}, STACK_SIZE={STACK_SIZE})")

In [ ]:
import time
from datetime import datetime

# Change this to your team ID. It is used in the submission file name.
TEAM_ID = "team00"

# The AI's play and the submission files are made in the same run.
# 1 run = 1 episode = 1 set of submission files (each run makes a new folder).
EPISODES = 1

RESULTS_DIR = Path("../results")
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
submission_dir = RESULTS_DIR / f"{TEAM_ID}_{RUN_TS}"
submission_dir.mkdir(parents=True, exist_ok=True)

ACTION_NAMES = ["_".join(a) if a else "NOOP" for a in SIMPLE_MOVEMENT]


frame_rows = []

fps = env.metadata.get("render_fps", 60)
frame_duration = 1 / fps

for episode in range(EPISODES):
    obs, info = env.reset()
    stacker = FrameStacker(STACK_SIZE)
    stack = stacker.reset(preprocess_frame(obs))

    video_path = submission_dir / f"ep{episode:02d}.mp4"
    h, w = obs.shape[0], obs.shape[1]
    video_writer = cv2.VideoWriter(
        str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h)
    )

    steps = 0
    frame_idx = 0
    terminated = truncated = False
    while not (terminated or truncated):
        stack_tensor = torch.from_numpy(stack).unsqueeze(0).to(device)
        with torch.no_grad():
            action = model(stack_tensor).argmax(dim=1).item()

        for _ in range(RECORD_EVERY_N):
            t0 = time.time()

            obs, reward, terminated, truncated, info = env.step(action)
            steps += 1

            env.render()
            video_writer.write(cv2.cvtColor(obs, cv2.COLOR_RGB2BGR))

            frame_rows.append({
                "episode": episode,
                "frame_idx": frame_idx,
                "action": action,
                "action_name": ACTION_NAMES[action],
                "x_pos": info["x_pos"],
                "y_pos": info["y_pos"],
                "x_pos_max": info["x_pos_max"],
                "coins": info["coins"],
                "score": info["score"],
                "time_remaining": info["time"],
                "flag_get": info["flag_get"],
                "status": info["status"],
                "life": info["life"],
            })
            frame_idx += 1

            elapsed = time.time() - t0
            if elapsed < frame_duration:
                time.sleep(frame_duration - elapsed)

            if terminated or truncated:
                break

        stack = stacker.step(preprocess_frame(obs))

    video_writer.release()

    print(
        f"episode {episode}: score={info['score']} x_pos_max={info['x_pos_max']} "
        f"coins={info['coins']} flag_get={info['flag_get']} steps={steps} "
        f"video={video_path.name}"
    )

env.close()

## Creating the submission files

The video was already written directly to `results/<TEAM_ID>_<timestamp>/` during play.
This cell adds two CSV files there and packs everything, together with the checkpoint and this notebook, into a zip file to submit.

| File | Contents |
|---|---|
| `epNN.mp4` | Video of the AI's play |
| `frames.csv` | A log **for every frame**: Mario's position (`x_pos`, `y_pos`), the chosen action, coins, score, time left, goal reached (`flag_get`), and more |
| `episodes.csv` | A summary for each episode and the **Skill score** |
| `<checkpoint>.pt` | The model that played. Replaying it reproduces the same episode, so the result can be checked |
| `04_playagent.ipynb` | This notebook, as it was run. The model class and the play loop are defined here, so it is needed to replay the episode |

**Save this notebook (Ctrl+S) before you run the next cell.** The zip gets the notebook as it is saved on disk.
If the saved file does not match the cells that ran, the next cell stops with an error and does not make the zip.

In [ ]:
import json
import zipfile

import pandas as pd

# The formula and its constants live in competition_config.py (shared with 01/02)
from competition_config import X_GOAL, add_skill_score

# 1. the per-frame log
frames_df = pd.DataFrame(frame_rows)
frames_df.to_csv(submission_dir / "frames.csv", index=False, encoding="utf-8")

# 2. per-episode summary, derived from the per-frame log so the two can never
#    disagree. x_pos_max/coins/score only grow and time_remaining only falls, so
#    max/min over the episode is the value at the end of it.
per_episode = (
    frames_df.groupby("episode")
    .agg(
        num_frames=("frame_idx", "count"),
        x_pos_max=("x_pos_max", "max"),
        coins=("coins", "max"),
        score=("score", "max"),
        time_remaining=("time_remaining", "min"),
        flag_get=("flag_get", "max"),
    )
    .reset_index()
)
episodes_df = add_skill_score(per_episode)
episodes_df["checkpoint"] = CHECKPOINT_PATH.name
episodes_df["stack_size"] = STACK_SIZE
episodes_df.to_csv(submission_dir / "episodes.csv", index=False, encoding="utf-8")

# 3. this notebook itself. The model class, preprocessing and play loop are
#    defined here, so replaying the episode needs this exact file as well as
#    the checkpoint. VS Code tells the kernel where the notebook file is; other
#    front ends fall back to the default name in the current folder.
NOTEBOOK_PATH = Path(globals().get("__vsc_ipynb_file__", "04_playagent.ipynb"))
saved_cells = [
    "".join(cell["source"]).strip()
    for cell in json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))["cells"]
    if cell["cell_type"] == "code"
]
# The zip gets the notebook as saved on disk, not what is in the editor, so
# check that the two agree. Every saved code cell must be one that ran in this
# session (In holds the code of every cell run so far), and the saved file must
# hold this run's checkpoint and team ID: a cell edited and re-run without
# saving still matches its older run in In, so the first check alone misses it.
ran = {code.strip() for code in In}
not_run = [i for i, code in enumerate(saved_cells, start=1) if code and code not in ran]
saved_text = "\n".join(saved_cells)
missing = [v for v in (CHECKPOINT_PATH.name, TEAM_ID) if v not in saved_text]
if not_run or missing:
    raise RuntimeError(
        f"{NOTEBOOK_PATH.name} as saved on disk does not match this run "
        f"(code cells not run as saved, counting from 1: {not_run}, values not saved: {missing}). "
        "Save the notebook (Ctrl+S), run all cells again, and then run this cell."
    )

# 4. one zip to hand in (the videos are already sitting in submission_dir).
#    The checkpoint goes in too: the play is deterministic, so replaying it
#    reproduces this exact episode and lets the result be verified.
videos = sorted(submission_dir.glob("*.mp4"))
zip_path = submission_dir.parent / f"submission_{TEAM_ID}_{RUN_TS}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(submission_dir.rglob("*")):
        zf.write(path, path.relative_to(submission_dir.parent))
    zf.write(CHECKPOINT_PATH, f"{submission_dir.name}/{CHECKPOINT_PATH.name}")
    zf.write(NOTEBOOK_PATH, f"{submission_dir.name}/{NOTEBOOK_PATH.name}")

print(f"=== submission for {TEAM_ID} ===")
print(f"frames.csv : {len(frames_df)} rows x {len(frames_df.columns)} columns")
print(f"videos     : {len(videos)}")
print(f"checkpoint : {CHECKPOINT_PATH.name}")
print(f"notebook   : {NOTEBOOK_PATH.name}")
print(f"distance   : {episodes_df['x_pos_max'].mean():.0f} / {X_GOAL}")
print(f"cleared    : {'yes' if episodes_df['clear'].any() else 'no'}")
print(f"skill      : {episodes_df['skill'].mean():.1f}")
print(f"\nsubmit this file: {zip_path}  ({zip_path.stat().st_size / 1024**2:.1f} MB)")

episodes_df[["episode", "num_frames", "x_pos_max", "coins", "score", "time_remaining",
             "flag_get", "progress", "speed", "points", "skill"]].round(2)